# Experiment: Case-001 Mitapivat Jurisdiction Access Verification Gate

Objective: separate foreign approvals, registry hits, query-adjacent records, and local access claims before clinician inquiry.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

registry = json.loads(
    Path(
        "data/registries/clinicaltrials/2026-05-12-active-mitapivat-thalassemia.json"
    ).read_text()
)
jurisdiction_map = json.loads(
    Path(
        "data/regulatory/mitapivat/2026-05-12-mitapivat-jurisdiction-access-map.json"
    ).read_text()
)


def record_domain(title: str) -> str:
    """Return a public-safe domain label for a registry record title."""
    text = title.lower()
    if "sickle cell" in text:
        return "query_adjacent_sickle_cell"
    if "non-transfusion" in text or "ntdt" in text:
        return "thalassemia_ntdt_context"
    if "transfusion-dependent" in text or "tdt" in text:
        return "thalassemia_tdt_context"
    if "thalassemia" in text:
        return "thalassemia_unclear_context"
    return "query_adjacent_context"


registry_domains = {
    record["nct_id"]: record_domain(record["title"]) for record in registry["records"]
}
direct_records = [
    nct_id
    for nct_id, domain in registry_domains.items()
    if domain.startswith("thalassemia_")
]
query_adjacent_records = [
    nct_id
    for nct_id, domain in registry_domains.items()
    if domain.startswith("query_adjacent")
]

decision = {
    "gate_label": "mitapivat_jurisdiction_access_verification_ready",
    "registry_records_checked": len(registry["records"]),
    "direct_thalassemia_records": direct_records,
    "query_adjacent_records": query_adjacent_records,
    "jurisdictions_checked": [
        record["jurisdiction"] for record in jurisdiction_map["records"]
    ],
    "policy": "foreign_approval_or_registry_presence_is_not_local_access",
}

decision

## Decision

Continue with jurisdiction access verification. Foreign approvals, registry hits, and local access claims must stay separate until qualified owner review.